In [1]:
import os, json, random
import numpy as np
import pandas as pd
import math
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Load Data

In [4]:
data = torch.load("/kaggle/input/datasets/teamcatalyst/initial-dataset/torch_tensor_words.pt", map_location="cpu")

In [5]:
title_ids   = data["title_ids"]
title_mask  = data["title_mask"]
tag_ids     = data["tag_ids"]
tag_mask    = data["tag_mask"]

member_text_ids  = data["member_text_ids"]
member_text_mask = data["member_text_mask"]
hard_ids   = data["hard_ids"]
hard_mask  = data["hard_mask"]
soft_ids   = data["soft_ids"]
soft_mask  = data["soft_mask"]
int_ids    = data["int_ids"]
int_mask   = data["int_mask"]

In [6]:
member_ids_df  = pd.read_csv("/kaggle/input/datasets/teamcatalyst/initial-dataset/member_ids.csv")
project_ids_df = pd.read_csv("/kaggle/input/datasets/teamcatalyst/initial-dataset/project_ids.csv")

In [7]:
member_ids_list  = member_ids_df["member_id"].astype(int).tolist()
project_ids_list = project_ids_df["project_id"].astype(int).tolist()

member_id2idx  = {mid: i for i, mid in enumerate(member_ids_list)}
project_id2idx = {pid: i for i, pid in enumerate(project_ids_list)}

num_members  = len(member_ids_list)
num_projects = len(project_ids_list)

print("Members:", num_members, "Projects:", num_projects)

Members: 38462 Projects: 18569


In [8]:
# sanity: tensor sizes must match id lists
assert member_text_ids.size(0) == num_members, "member tensor rows != member_ids rows"
assert title_ids.size(0) == num_projects, "project tensor rows != project_ids rows"

In [9]:
edges = pd.read_csv("/kaggle/input/datasets/teamcatalyst/initial-dataset/B_graph.csv")

In [10]:
edges["member_id"]  = edges["member_id"].astype(int)
edges["project_id"] = edges["project_id"].astype(int)

# remove duplicate interactions
edges = edges.drop_duplicates(subset=["member_id", "project_id"]).reset_index(drop=True)
print("Total unique edges:", len(edges))

Total unique edges: 66553


In [11]:
edges_shuffled = edges.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_total = len(edges_shuffled)
n_train = int(0.8 * n_total)

train_edges = edges_shuffled.iloc[:n_train].reset_index(drop=True)
test_edges  = edges_shuffled.iloc[n_train:].reset_index(drop=True)

print("Train edges:", len(train_edges))
print("Test edges :", len(test_edges))
print("Train ratio:", len(train_edges) / n_total)

Train edges: 53242
Test edges : 13311
Train ratio: 0.7999939897525281


In [12]:
train_set = set(map(tuple, train_edges[["member_id","project_id"]].values))
test_set  = set(map(tuple, test_edges[["member_id","project_id"]].values))
print("Edge overlap:", len(train_set & test_set))

print("Unique members - train:", train_edges["member_id"].nunique(), "| test:", test_edges["member_id"].nunique())
print("Unique projects - train:", train_edges["project_id"].nunique(), "| test:", test_edges["project_id"].nunique())

Edge overlap: 0
Unique members - train: 33621 | test: 11495
Unique projects - train: 18487 | test: 10162


In [13]:
# save split files (optional)
train_edges.to_csv("/kaggle/working/train_edges.csv", index=False)
test_edges.to_csv("/kaggle/working/test_edges.csv", index=False)
print("Saved: /kaggle/working/train_edges.csv and /kaggle/working/test_edges.csv")

Saved: /kaggle/working/train_edges.csv and /kaggle/working/test_edges.csv


In [14]:
# Map original IDs -> internal indices
train_members_mapped  = train_edges["member_id"].map(member_id2idx)
train_projects_mapped = train_edges["project_id"].map(project_id2idx)

valid_train = train_members_mapped.notna() & train_projects_mapped.notna()
train_members  = train_members_mapped[valid_train].astype(int).to_numpy()
train_projects = train_projects_mapped[valid_train].astype(int).to_numpy()

test_members_mapped  = test_edges["member_id"].map(member_id2idx)
test_projects_mapped = test_edges["project_id"].map(project_id2idx)

valid_test = test_members_mapped.notna() & test_projects_mapped.notna()
test_members  = test_members_mapped[valid_test].astype(int).to_numpy()
test_projects = test_projects_mapped[valid_test].astype(int).to_numpy()

print("Mapped train pairs:", train_members.shape, train_projects.shape)
print("Mapped test pairs :", test_members.shape, test_projects.shape)

Mapped train pairs: (53242,) (53242,)
Mapped test pairs : (13311,) (13311,)


In [15]:
# extra sanity checks (catch mapping bugs)
assert train_members.min() >= 0 and train_members.max() < num_members
assert train_projects.min() >= 0 and train_projects.max() < num_projects
assert test_members.min() >= 0 and test_members.max() < num_members
assert test_projects.min() >= 0 and test_projects.max() < num_projects

In [16]:
train_pos_by_project = [set() for _ in range(num_projects)]
for u_idx, v_idx in zip(train_members, train_projects):
    train_pos_by_project[int(v_idx)].add(int(u_idx))

print("Avg train developers per project:", np.mean([len(s) for s in train_pos_by_project]))
print("Projects with 0 train developers:", sum(1 for s in train_pos_by_project if len(s) == 0))

Avg train developers per project: 2.8672518713985675
Projects with 0 train developers: 82


In [17]:
train_pos_by_user = [set() for _ in range(num_members)]
for u_idx, v_idx in zip(train_members, train_projects):
    train_pos_by_user[int(u_idx)].add(int(v_idx))

print("Avg train projects per dev:", np.mean([len(s) for s in train_pos_by_user]))
print("Devs with 0 train projects:", sum(1 for s in train_pos_by_user if len(s) == 0))

Avg train projects per dev: 1.3842753886953356
Devs with 0 train projects: 4841


In [18]:
# (recommended for evaluation sampling later)
test_pos_by_project = [set() for _ in range(num_projects)]
for u_idx, v_idx in zip(test_members, test_projects):
    test_pos_by_project[int(v_idx)].add(int(u_idx))

print("Avg test developers per project:", np.mean([len(s) for s in test_pos_by_project]))

Avg test developers per project: 0.7168398944477354


## One-Hop Functions

In [19]:
def ensure_nonempty_sequence(ids_b, mask_b, unk_id=1):
    empty = ~mask_b.any(dim=1)
    if empty.any():
        ids_b = ids_b.clone()
        mask_b = mask_b.clone()
        ids_b[empty, 0] = unk_id
        mask_b[empty, 0] = True
    return ids_b, mask_b

In [20]:
# makes each word vector smarter (context-aware)
# so the model understands relationships between words in the title before summarizing it into one project vector.
# Eq.5 in CSDRec paper
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d: int, n_heads: int):
        super().__init__()
        self.d = d
        self.n_heads = n_heads
        self.w1 = nn.Parameter(torch.empty(n_heads, d, d))
        self.w2 = nn.Parameter(torch.empty(n_heads, d, d))
        nn.init.xavier_uniform_(self.w1)
        nn.init.xavier_uniform_(self.w2)

    def forward(self, E: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask = mask.bool()
        B, L, D = E.shape
        neg_inf = torch.finfo(E.dtype).min

        head_outs = []
        for k in range(self.n_heads):
            W1 = self.w1[k]  # [D,D]
            W2 = self.w2[k]  # [D,D]

            EW1 = torch.matmul(E, W1)                         # [B,L,D]
            scores = torch.matmul(EW1, E.transpose(1, 2))      # [B,L,L]

            key_mask = mask.unsqueeze(1).expand(B, L, L)       # mask keys j
            scores = scores.masked_fill(~key_mask, neg_inf)

            A = torch.softmax(scores, dim=2)                   # over j
            ctx = torch.matmul(A, E)                           # [B,L,D]
            h_k = torch.matmul(ctx, W2.transpose(0, 1))         # [B,L,D]
            head_outs.append(h_k)

        H = torch.cat(head_outs, dim=2)                         # [B,L,n_heads*D]
        H = H.masked_fill(~mask.unsqueeze(-1), 0.0)              # zero padded queries i
        return H

In [21]:
# decides which words matter most and summarizes them into one vector
# Eq.6 in CSDRec paper
class AdditiveAttention(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.b = nn.Linear(dim, dim, bias=True)     # b_w x + c_w
        self.a = nn.Parameter(torch.empty(dim))     # a_w
        nn.init.xavier_uniform_(self.b.weight)
        nn.init.zeros_(self.b.bias)
        nn.init.normal_(self.a, std=0.02)

    def forward(self, X: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask = mask.bool()
        B, L, D = X.shape

        H = torch.tanh(self.b(X))                  # [B,L,D]
        scores = torch.matmul(H, self.a)           # [B,L]

        valid_row = mask.any(dim=1)                # [B]
        out = torch.zeros((B, D), device=X.device, dtype=X.dtype)
        if valid_row.any():
            scores_v = scores[valid_row]
            mask_v   = mask[valid_row]
            neg_inf = torch.finfo(scores_v.dtype).min
            scores_v = scores_v.masked_fill(~mask_v, neg_inf)
            alpha = torch.softmax(scores_v, dim=1)
            out[valid_row] = torch.bmm(alpha.unsqueeze(1), X[valid_row]).squeeze(1)
        return out

In [22]:
# Eq.8 in CSDRec paper
class Encoder(nn.Module):
    """
    Variant: task title uses Eq(5)+Eq(6), dev text ALSO uses Eq(5)+Eq(6),
    labels use Eq(6).
    """
    def __init__(self, word_vocab_size: int, label_vocab_size: int,
                 d_word: int = 128, d_label: int = 128, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.word_emb  = nn.Embedding(word_vocab_size, d_word, padding_idx=0)
        self.label_emb = nn.Embedding(label_vocab_size, d_label, padding_idx=0)
        self.dropout = nn.Dropout(dropout)

        self.eq5_words = MultiHeadSelfAttention(d=d_word, n_heads=n_heads)

        self.word_pool  = AdditiveAttention(dim=n_heads * d_word)  # pools Eq(5) outputs
        self.label_pool = AdditiveAttention(dim=d_label)

        self.n_heads = n_heads
        self.d_word = d_word
        self.d_label = d_label

    def encode_project(self, title_ids, title_mask, tag_ids, tag_mask):
        title_ids, title_mask = ensure_nonempty_sequence(title_ids, title_mask, unk_id=1)

        E = self.dropout(self.word_emb(title_ids))          # [B,L,Dw]
        H = self.eq5_words(E, title_mask)                   # [B,L,nH*Dw]
        vtitle = self.word_pool(H, title_mask)              # [B,nH*Dw]

        Lemb = self.dropout(self.label_emb(tag_ids))        # [B,Lt,Dl]
        vlabel = self.label_pool(Lemb, tag_mask)            # [B,Dl]

        return torch.cat([vtitle, vlabel], dim=1)           # [B, nH*Dw + Dl]

    def encode_member(self, text_ids, text_mask, devlbl_ids, devlbl_mask):
        text_ids, text_mask = ensure_nonempty_sequence(text_ids, text_mask, unk_id=1)

        E = self.dropout(self.word_emb(text_ids))           # [B,L,Dw]
        H = self.eq5_words(E, text_mask)                    # [B,L,nH*Dw]
        utext = self.word_pool(H, text_mask)                # [B,nH*Dw]

        Lemb = self.dropout(self.label_emb(devlbl_ids))     # [B,Ll,Dl]
        ulabel = self.label_pool(Lemb, devlbl_mask)         # [B,Dl]

        return torch.cat([utext, ulabel], dim=1)            # [B, nH*Dw + Dl]

In [23]:
class OneHopModule(nn.Module):
    def __init__(self, word_vocab_size, label_vocab_size,
                 num_devs, num_tasks,
                 d_word=128, d_label=128, n_heads=4,
                 d_id=64, common_dim=256, dropout=0.1):
        super().__init__()
        self.d_id = d_id

        self.text = Encoder(
            word_vocab_size, label_vocab_size,
            d_word=d_word, d_label=d_label, n_heads=n_heads, dropout=dropout
        )

        self.dev_id  = nn.Embedding(num_devs,  d_id)
        self.task_id = nn.Embedding(num_tasks, d_id)

        task_sem_dim = (n_heads * d_word) + d_label
        dev_sem_dim  = (n_heads * d_word) + d_label  # text + labels

        self.dev_proj  = nn.Linear(d_id + dev_sem_dim,  common_dim)
        self.task_proj = nn.Linear(d_id + task_sem_dim, common_dim)

    def score(self, dev_idx, task_idx,
              # developer text + merged labels
              text_ids, text_mask, devlbl_ids, devlbl_mask,
              # task title+tags
              title_ids, title_mask, tag_ids, tag_mask,
              dev_known_mask=None, task_known_mask=None):

        # semantic embeddings
        u_t = self.text.encode_member(text_ids, text_mask, devlbl_ids, devlbl_mask)
        v_t = self.text.encode_project(title_ids, title_mask, tag_ids, tag_mask)

        # ----- ID gating (cold-start) -----
        if dev_known_mask is None:
            u_id = self.dev_id(dev_idx)
        else:
            u_id = torch.zeros(
                (dev_idx.size(0), self.d_id),
                device=dev_idx.device,
                dtype=self.dev_id.weight.dtype
            )
            if dev_known_mask.any():
                u_id[dev_known_mask] = self.dev_id(dev_idx[dev_known_mask])

        if task_known_mask is None:
            v_id = self.task_id(task_idx)
        else:
            v_id = torch.zeros(
                (task_idx.size(0), self.d_id),
                device=task_idx.device,
                dtype=self.task_id.weight.dtype
            )
            if task_known_mask.any():
                v_id[task_known_mask] = self.task_id(task_idx[task_known_mask])

        # ----- project to common space + dot product -----
        z_u = self.dev_proj(torch.cat([u_id, u_t], dim=1))
        z_v = self.task_proj(torch.cat([v_id, v_t], dim=1))
        return (z_u * z_v).sum(dim=1)

In [24]:
def merge_dev_labels(h_ids_b, h_mask_b, s_ids_b, s_mask_b, i_ids_b, i_mask_b):
    devlbl_ids  = torch.cat([h_ids_b, s_ids_b, i_ids_b], dim=1)   # [B, Lh+Ls+Li]
    devlbl_mask = torch.cat([h_mask_b, s_mask_b, i_mask_b], dim=1)
    return devlbl_ids, devlbl_mask

In [25]:
with open("/kaggle/input/datasets/teamcatalyst/initial-dataset/word2id.json", "r") as f:
    word2id = json.load(f)
with open("/kaggle/input/datasets/teamcatalyst/initial-dataset/label2id.json", "r") as f:
    label2id = json.load(f)

In [26]:
model = OneHopModule(
    word_vocab_size=len(word2id),
    label_vocab_size=len(label2id),
    num_devs=num_members,
    num_tasks=num_projects,
    d_word=128,
    d_label=128,
    n_heads=4,
    d_id=64,
    common_dim=256,
    dropout=0.1
).to(device)

In [27]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)

In [28]:
def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

In [29]:
# tensor consistency checks (updated)
assert num_members == hard_ids.size(0) == soft_ids.size(0) == int_ids.size(0)
assert num_projects == title_ids.size(0) == tag_ids.size(0)

## Train One-Hop with BPR

In [30]:
pos_set_by_project = train_pos_by_project

In [31]:
def sample_negative_member(v_idx: int, max_tries: int = 1000) -> int:
    """Sample a negative developer index not in project's TRAIN positives."""
    pos = pos_set_by_project[v_idx]

    if len(pos) >= num_members:
        return random.randrange(num_members)

    for _ in range(max_tries):
        u = random.randrange(num_members)
        if u not in pos:
            return u

    # fallback
    candidates = list(set(range(num_members)) - pos)
    return random.choice(candidates) if candidates else random.randrange(num_members)

In [32]:
BATCH_SIZE = 512
EPOCHS = 100

train_pairs = list(zip(train_members.tolist(), train_projects.tolist()))

if device.startswith("cuda"):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# Training Loop
for epoch in range(1, EPOCHS + 1):

    random.shuffle(train_pairs)
    model.train()

    total_loss = 0.0
    num_batches = (len(train_pairs) + BATCH_SIZE - 1) // BATCH_SIZE
    pbar = tqdm(range(num_batches), desc=f"Epoch {epoch}/{EPOCHS}")

    for bi in pbar:
        batch = train_pairs[bi * BATCH_SIZE : (bi + 1) * BATCH_SIZE]
        if not batch:
            continue

        # Fix project, rank developers
        v_idx = torch.tensor([x[1] for x in batch], dtype=torch.long, device=device)
        u_pos = torch.tensor([x[0] for x in batch], dtype=torch.long, device=device)

        # sample negative developers
        u_neg_list = [sample_negative_member(int(v)) for v in v_idx.tolist()]
        u_neg = torch.tensor(u_neg_list, dtype=torch.long, device=device)

        # warm start during training
        v_known_mask = torch.ones_like(v_idx, dtype=torch.bool)
        upos_known_mask = torch.ones_like(u_pos, dtype=torch.bool)
        uneg_known_mask = torch.ones_like(u_neg, dtype=torch.bool)

        # Slice developer tensors (POS)
        up_cpu = u_pos.detach().cpu()

        mt_pos   = member_text_ids[up_cpu].to(device)
        mtm_pos  = member_text_mask[up_cpu].to(device)

        h_pos    = hard_ids[up_cpu].to(device)
        hm_pos   = hard_mask[up_cpu].to(device)

        s_pos    = soft_ids[up_cpu].to(device)
        sm_pos   = soft_mask[up_cpu].to(device)

        i_pos    = int_ids[up_cpu].to(device)
        im_pos   = int_mask[up_cpu].to(device)

        devlbl_pos, devlbl_mask_pos = merge_dev_labels(
            h_pos, hm_pos, s_pos, sm_pos, i_pos, im_pos
        )

        # Slice developer tensors (NEG)
        un_cpu = u_neg.detach().cpu()

        mt_neg   = member_text_ids[un_cpu].to(device)
        mtm_neg  = member_text_mask[un_cpu].to(device)

        h_neg    = hard_ids[un_cpu].to(device)
        hm_neg   = hard_mask[un_cpu].to(device)

        s_neg    = soft_ids[un_cpu].to(device)
        sm_neg   = soft_mask[un_cpu].to(device)

        i_neg    = int_ids[un_cpu].to(device)
        im_neg   = int_mask[un_cpu].to(device)

        devlbl_neg, devlbl_mask_neg = merge_dev_labels(
            h_neg, hm_neg, s_neg, sm_neg, i_neg, im_neg
        )

        # Slice project tensors (fixed v)
        v_cpu = v_idx.detach().cpu()

        t_ids_b  = title_ids[v_cpu].to(device)
        t_mask_b = title_mask[v_cpu].to(device)

        g_ids_b  = tag_ids[v_cpu].to(device)
        g_mask_b = tag_mask[v_cpu].to(device)

        # Optimization
        optimizer.zero_grad(set_to_none=True)

        # positive scores (u+, v)
        pos_scores = model.score(
            dev_idx=u_pos, task_idx=v_idx,
            text_ids=mt_pos, text_mask=mtm_pos,
            devlbl_ids=devlbl_pos, devlbl_mask=devlbl_mask_pos,
            title_ids=t_ids_b, title_mask=t_mask_b,
            tag_ids=g_ids_b, tag_mask=g_mask_b,
            dev_known_mask=upos_known_mask,
            task_known_mask=v_known_mask
        )

        # negative scores (u-, v)
        neg_scores = model.score(
            dev_idx=u_neg, task_idx=v_idx,
            text_ids=mt_neg, text_mask=mtm_neg,
            devlbl_ids=devlbl_neg, devlbl_mask=devlbl_mask_neg,
            title_ids=t_ids_b, title_mask=t_mask_b,
            tag_ids=g_ids_b, tag_mask=g_mask_b,
            dev_known_mask=uneg_known_mask,
            task_known_mask=v_known_mask
        )

        loss = bpr_loss(pos_scores, neg_scores)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        pbar.set_postfix(loss=total_loss / (bi + 1))

    print(f"Epoch {epoch} avg loss: {total_loss / max(1, num_batches):.6f}")

torch.save(model.state_dict(), "/kaggle/working/trained_one_hop_weights.pt")
print("Saved:", "/kaggle/working/trained_one_hop_weights.pt")

Epoch 1/100: 100%|██████████| 104/104 [00:22<00:00,  4.55it/s, loss=0.734]


Epoch 1 avg loss: 0.734444


Epoch 2/100: 100%|██████████| 104/104 [00:21<00:00,  4.79it/s, loss=0.641]


Epoch 2 avg loss: 0.641234


Epoch 3/100: 100%|██████████| 104/104 [00:21<00:00,  4.76it/s, loss=0.595]


...

Epoch 98 avg loss: 0.042804


Epoch 99/100: 100%|██████████| 104/104 [00:22<00:00,  4.60it/s, loss=0.0437]


Epoch 99 avg loss: 0.043707


Epoch 100/100: 100%|██████████| 104/104 [00:22<00:00,  4.59it/s, loss=0.0422]

Epoch 100 avg loss: 0.042250
Saved: /kaggle/working/trained_one_hop_weights.pt


## Save trained data

In [33]:
model.eval()

train_member_idx = np.unique(train_members).astype(int)
train_member_ids = np.array(
    [member_ids_list[i] for i in train_member_idx],
    dtype=np.int64
)

member_emb_chunks = []

with torch.no_grad():
    for i in range(0, len(train_member_idx), BATCH_SIZE):

        batch_idx = train_member_idx[i:i+BATCH_SIZE]
        idx_tensor = torch.tensor(batch_idx, dtype=torch.long, device=device)

        # ---- slice developer tensors ----
        mt  = member_text_ids[batch_idx].to(device)
        mtm = member_text_mask[batch_idx].to(device)

        h   = hard_ids[batch_idx].to(device)
        hm  = hard_mask[batch_idx].to(device)

        s   = soft_ids[batch_idx].to(device)
        sm  = soft_mask[batch_idx].to(device)

        i_  = int_ids[batch_idx].to(device)
        im  = int_mask[batch_idx].to(device)

        # merge labels
        devlbl_ids, devlbl_mask = merge_dev_labels(h, hm, s, sm, i_, im)

        # semantic embedding
        u_t = model.text.encode_member(mt, mtm, devlbl_ids, devlbl_mask)

        # ID embedding
        u_id = model.dev_id(idx_tensor)

        # final embedding in common space
        z_u = model.dev_proj(torch.cat([u_id, u_t], dim=1))

        member_emb_chunks.append(z_u.cpu())

train_member_embs = torch.cat(member_emb_chunks, dim=0).numpy()

np.save("/kaggle/working/train_member_embeddings.npy", train_member_embs)
pd.DataFrame({"member_id": train_member_ids}).to_csv(
    "/kaggle/working/train_member_ids.csv",
    index=False
)

print("Saved train members:", train_member_embs.shape)

Saved train members: (33621, 256)


In [34]:
train_project_idx = np.unique(train_projects).astype(int)
train_project_ids = np.array(
    [project_ids_list[i] for i in train_project_idx],
    dtype=np.int64
)

project_emb_chunks = []

with torch.no_grad():
    for i in range(0, len(train_project_idx), BATCH_SIZE):

        batch_idx = train_project_idx[i:i+BATCH_SIZE]
        idx_tensor = torch.tensor(batch_idx, dtype=torch.long, device=device)

        # slice project tensors
        ti = title_ids[batch_idx].to(device)
        tm = title_mask[batch_idx].to(device)

        gi = tag_ids[batch_idx].to(device)
        gm = tag_mask[batch_idx].to(device)

        # semantic embedding
        v_t = model.text.encode_project(ti, tm, gi, gm)

        # ID embedding
        v_id = model.task_id(idx_tensor)

        # final embedding in common space
        z_v = model.task_proj(torch.cat([v_id, v_t], dim=1))

        project_emb_chunks.append(z_v.cpu())

train_project_embs = torch.cat(project_emb_chunks, dim=0).numpy()

np.save("/kaggle/working/train_project_embeddings.npy", train_project_embs)
pd.DataFrame({"project_id": train_project_ids}).to_csv(
    "/kaggle/working/train_project_ids.csv",
    index=False
)

print("Saved train projects:", train_project_embs.shape)

Saved train projects: (18487, 256)


## Evaluation

In [35]:
def sample_eval_candidates_users(v_idx, u_true, num_neg=99):
    """
    Sample candidate developers for a fixed project.
    Ensure negatives are not train/test positives.
    """
    v_idx = int(v_idx)
    u_true = int(u_true)

    forbid = train_pos_by_project[v_idx] | test_pos_by_project[v_idx]

    candidates = [u_true]
    seen = {u_true}

    while len(candidates) < 1 + num_neg:
        u = np.random.randint(0, num_members)
        if u in seen or u in forbid:
            continue
        candidates.append(u)
        seen.add(u)

    return candidates

In [36]:
def rank_metrics(ranks, K=10):
    """
    ranks: list of 1-based ranks of the true developer (1 means best).
    """
    hit = np.mean([1.0 if r <= K else 0.0 for r in ranks])
    ndcg = np.mean([1.0 / math.log2(r + 1) if r <= K else 0.0 for r in ranks])
    return hit, ndcg

In [37]:
# Build positive sets (already built in training, but safe to rebuild)
train_pos_by_project = [set() for _ in range(num_projects)]
for u, v in zip(train_members, train_projects):
    train_pos_by_project[int(v)].add(int(u))

test_pos_by_project = [set() for _ in range(num_projects)]
for u, v in zip(test_members, test_projects):
    test_pos_by_project[int(v)].add(int(u))

In [38]:
K = 48
NUM_NEG = 99

# Known sets for cold-start handling
train_user_known = set(train_members.tolist())
train_proj_known = set(train_projects.tolist())

# Main Evaluation Loop
true_ranks = []
model.eval()

with torch.no_grad():

    for (u_true, v) in tqdm(
        list(zip(test_members, test_projects)),
        desc="Evaluation"
    ):

        # Sample candidate developers
        cand_users = sample_eval_candidates_users(v, u_true, num_neg=NUM_NEG)

        # Tensors
        u_idx = torch.tensor(cand_users, dtype=torch.long, device=device)
        v_idx = torch.tensor([v] * len(cand_users), dtype=torch.long, device=device)

        # Slice Developer Tensors
        u_cpu = u_idx.cpu()

        mt_ids  = member_text_ids[u_cpu].to(device)
        mt_mask = member_text_mask[u_cpu].to(device)

        h_ids  = hard_ids[u_cpu].to(device)
        h_mask = hard_mask[u_cpu].to(device)

        s_ids  = soft_ids[u_cpu].to(device)
        s_mask = soft_mask[u_cpu].to(device)

        i_ids  = int_ids[u_cpu].to(device)
        i_mask = int_mask[u_cpu].to(device)

        devlbl_ids, devlbl_mask = merge_dev_labels(
            h_ids, h_mask,
            s_ids, s_mask,
            i_ids, i_mask
        )

        # Slice Project Tensors
        v_cpu = torch.tensor([v]).cpu()

        t_ids  = title_ids[v_cpu].to(device).repeat(len(cand_users), 1)
        t_mask = title_mask[v_cpu].to(device).repeat(len(cand_users), 1)

        g_ids  = tag_ids[v_cpu].to(device).repeat(len(cand_users), 1)
        g_mask = tag_mask[v_cpu].to(device).repeat(len(cand_users), 1)

        # Cold Start Masks
        dev_known_mask = torch.tensor(
            [u in train_user_known for u in cand_users],
            dtype=torch.bool,
            device=device
        )

        task_known_mask = torch.tensor(
            [v in train_proj_known] * len(cand_users),
            dtype=torch.bool,
            device=device
        )

        # Compute Scores
        scores = model.score(
            dev_idx=u_idx,
            task_idx=v_idx,
            text_ids=mt_ids,
            text_mask=mt_mask,
            devlbl_ids=devlbl_ids,
            devlbl_mask=devlbl_mask,
            title_ids=t_ids,
            title_mask=t_mask,
            tag_ids=g_ids,
            tag_mask=g_mask,
            dev_known_mask=dev_known_mask,
            task_known_mask=task_known_mask
        )

        scores = scores.detach().cpu().numpy()

        # Rank (true developer is at index 0)
        order = np.argsort(-scores)
        rank = int(np.where(order == 0)[0][0]) + 1
        true_ranks.append(rank)

# Final Metrics
hit10, ndcg10 = rank_metrics(true_ranks, K=K)

print(f"Candidates: 1+{NUM_NEG} | Evaluated: {len(true_ranks)}")
print(f"Hit@{K}:  {hit10:.4f}")
print(f"NDCG@{K}: {ndcg10:.4f}")
print("Mean rank:", np.mean(true_ranks))
print("Median rank:", np.median(true_ranks))

Evaluation: 100%|██████████| 13311/13311 [02:30<00:00, 88.40it/s]

Candidates: 1+99 | Evaluated: 13311
Hit@48:  0.7178
NDCG@48: 0.2462
Mean rank: 33.46330102922395
Median rank: 28.0
